In [3]:
import os

# Maak powerbi folder aan als die niet bestaat
os.makedirs('../data/powerbi', exist_ok=True)

print("✅ PowerBI folder created/verified!")

✅ PowerBI folder created/verified!


In [4]:
import pandas as pd
import numpy as np
import os

# Maak powerbi folder aan
os.makedirs('../data/powerbi', exist_ok=True)
print("✅ PowerBI folder created/verified!")

print("="*60)
print("PREPARING DATA FOR POWER BI")
print("="*60)

# Laad de complete dataset
df = pd.read_csv('../data/processed/transactions_with_anomalies.csv')
df['Date'] = pd.to_datetime(df['Date'])

print(f"Total transactions: {len(df)}")
print(f"Date range: {df['Date'].min().date()} to {df['Date'].max().date()}")

# Selecteer en hernoem kolommen voor Power BI
powerbi_df = df[[
    'Date',
    'Amount',
    'Balance',
    'Counterparty',
    'Final_Category',
    'Is_Income',
    'Is_Expense',
    'Amount_Abs',
    'Year',
    'Month',
    'Month_Name',
    'Day',
    'Day_Name',
    'Is_Weekend',
    'Quarter',
    'Is_Anomaly_Either',
    'Prediction_Confidence'
]].copy()

# Hernoem kolommen
powerbi_df.columns = [
    'Date',
    'Amount',
    'Balance',
    'Merchant',
    'Category',
    'Is_Income',
    'Is_Expense',
    'Amount_Absolute',
    'Year',
    'Month_Number',
    'Month_Name',
    'Day',
    'Day_Name',
    'Is_Weekend',
    'Quarter',
    'Is_Anomaly',
    'ML_Confidence'
]

# Extra kolommen
powerbi_df['Year_Month'] = powerbi_df['Date'].dt.to_period('M').astype(str)
powerbi_df['Week_Number'] = powerbi_df['Date'].dt.isocalendar().week
powerbi_df['Transaction_Type'] = powerbi_df.apply(
    lambda x: 'Income' if x['Is_Income'] == 1 else 'Expense', axis=1
)
powerbi_df['Anomaly_Status'] = powerbi_df['Is_Anomaly'].apply(
    lambda x: 'Anomaly' if x == 1 else 'Normal'
)

print("\n✅ Data prepared for Power BI!")
print(f"Final shape: {powerbi_df.shape}")

# Save
powerbi_df.to_csv('../data/powerbi/transactions_powerbi.csv', index=False)
print(f"✅ Saved: transactions_powerbi.csv")

print("\nSample:")
print(powerbi_df.head(3))

✅ PowerBI folder created/verified!
PREPARING DATA FOR POWER BI
Total transactions: 1505
Date range: 2024-12-08 to 2025-12-08

✅ Data prepared for Power BI!
Final shape: (1505, 21)
✅ Saved: transactions_powerbi.csv

Sample:
        Date  Amount  Balance                           Merchant     Category  \
0 2024-12-08  -25.00   180.36             CCV*Cuijkse Brouwbriga  other_shops   
1 2024-12-08  -60.00    20.36  R. Pittens via Rabo Betaalverzoek    transport   
2 2024-12-09   -1.74    18.62                  Albert Heijn 1382  supermarket   

   Is_Income  Is_Expense  Amount_Absolute  Year  Month_Number  ... Day  \
0          0           1            25.00  2024            12  ...   8   
1          0           1            60.00  2024            12  ...   8   
2          0           1             1.74  2024            12  ...   9   

   Day_Name Is_Weekend  Quarter  Is_Anomaly  ML_Confidence  Year_Month  \
0    Sunday          1        4           0       0.680000     2024-12   
1    Su

In [5]:
print("="*60)
print("PREPARING PREDICTIONS FOR POWER BI")
print("="*60)

# Laad predictions
predictions = pd.read_csv('../data/processed/spending_predictions.csv')

# Selecteer en hernoem
predictions_powerbi = predictions[[
    'Category',
    'Predicted_Next_Month',
    'Historical_Average',
    'Last_Month_Actual'
]].copy()

predictions_powerbi.columns = [
    'Category',
    'Predicted_Amount',
    'Historical_Average',
    'Last_Month'
]

# Bereken verschil
predictions_powerbi['Change_vs_Average'] = (
    (predictions_powerbi['Predicted_Amount'] / predictions_powerbi['Historical_Average'] - 1) * 100
)

print(f"✅ Predictions prepared! ({len(predictions_powerbi)} categories)")

# Save
predictions_powerbi.to_csv('../data/powerbi/predictions_powerbi.csv', index=False)
print(f"✅ Saved: predictions_powerbi.csv")

print("\nSample:")
print(predictions_powerbi.head(3))

PREPARING PREDICTIONS FOR POWER BI
✅ Predictions prepared! (10 categories)
✅ Saved: predictions_powerbi.csv

Sample:
    Category  Predicted_Amount  Historical_Average  Last_Month  \
0       rent        423.696970          411.333333      417.00   
1   bar_cafe        426.036212          355.856667      591.09   
2  transport        330.873846          272.032308       64.88   

   Change_vs_Average  
0           3.005746  
1          19.721296  
2          21.630349  


In [8]:
import locale

print("="*60)
print("CREATING SUMMARY STATISTICS FOR POWER BI")
print("="*60)

# Force Engels formaat (punt als decimaal)
locale.setlocale(locale.LC_NUMERIC, 'C')

# Gebruik data uit geheugen of laad opnieuw
try:
    test = powerbi_df.shape
    print("Using data from memory")
except:
    print("Loading data from file")
    powerbi_df = pd.read_csv('../data/powerbi/transactions_powerbi.csv')
    powerbi_df['Date'] = pd.to_datetime(powerbi_df['Date'])

# Bereken metrics
total_transactions = len(powerbi_df)
total_income = powerbi_df[powerbi_df['Is_Income'] == 1]['Amount'].sum()
total_expenses = powerbi_df[powerbi_df['Is_Expense'] == 1]['Amount_Absolute'].sum()
net_cashflow = total_income - total_expenses
current_balance = powerbi_df.sort_values('Date')['Balance'].iloc[-1]
avg_transaction = powerbi_df['Amount_Absolute'].mean()
total_anomalies = powerbi_df['Is_Anomaly'].sum()
unique_merchants = powerbi_df['Merchant'].nunique()

# Maak dataframe
summary_stats = pd.DataFrame({
    'Metric': [
        'Total Transactions',
        'Total Income',
        'Total Expenses',
        'Net Cashflow',
        'Current Balance',
        'Average Transaction',
        'Total Anomalies',
        'Unique Merchants'
    ],
    'Value': [
        total_transactions,
        total_income,
        total_expenses,
        net_cashflow,
        current_balance,
        avg_transaction,
        total_anomalies,
        unique_merchants
    ]
})

print("\n" + "="*60)
print("SUMMARY TABLE")
print("="*60)
print(summary_stats)

# Save met punt als decimaal (Engels formaat)
summary_stats.to_csv('../data/powerbi/summary_stats.csv', 
                     index=False, 
                     decimal='.',
                     sep=',')

print(f"\n✅ Saved: summary_stats.csv (with decimal='.')")

print("\n" + "="*60)
print("ALL FILES CREATED!")
print("="*60)

CREATING SUMMARY STATISTICS FOR POWER BI
Using data from memory

SUMMARY TABLE
                Metric         Value
0   Total Transactions   1505.000000
1         Total Income  29750.970000
2       Total Expenses  29611.900000
3         Net Cashflow    139.070000
4      Current Balance     38.570000
5  Average Transaction     39.443767
6      Total Anomalies     76.000000
7     Unique Merchants    388.000000

✅ Saved: summary_stats.csv (with decimal='.')

ALL FILES CREATED!
